<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/01.%20Dataset%20Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Dataset Manager**
> MARC - Multidisciplinary AI Research Centre | UOP


This notebook serves as the central command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)
* **💻 Project Code:** [Wearable-Biomechanics Repository](https://github.com/janithcyapa/Wearable-Biomechanics/)
* **📂 Data Storage:** [Project Google Drive Folder](https://drive.google.com/drive/folders/1QERF02T4pvubAEjnv6f1VrCOWNYxHv54?usp=drive_link)


## 📚 Initial Setup & Configuration


### 🔧 One-Time Setup Instructions

#### **Step 1: Get Access to the Shared Dataset Folder**
1. Ask your **admin** to share the Google Drive folder with you
2. You'll receive an email with a "Shared with you" notification
3. Open the shared folder in Google Drive
4. The folder should be named: **`Wearable-Biomechanics`**

#### **Step 2: Add Shortcut to Your Drive**
1. In Google Drive, locate the shared folder under **"Shared with me"**
2. **Right-click** on the folder named `Wearable-Biomechanics`
3. Select **"Add shortcut to Drive"** (or **"Organize → Add shortcut"**)
4. Navigate to your **`My Drive > Datasets`** folder
   - If `Datasets` folder doesn't exist, create it first
5. Click **"Add Shortcut"**

#### **Step 3: Verify the Setup**
- Go to **`My Drive > Datasets`** in your Google Drive
- You should see **`Wearable-Biomechanics`** with a shortcut icon ⤴️
- The full path will be: **`MyDrive/Datasets/Wearable-Biomechanics`**


### ⚠️ Important Notes
- 🔑 **Permissions**: You need at least **"Editor"** access to upload data
- 🔗 **Shortcuts**: Always use the shortcut in your Drive, not the original shared folder
- 📂 **Structure**: The system will auto-create `/index`, `/raw_data`, and `/processed_data` subfolders
- 🚫 **Don't delete** the shortcut or you'll lose access to shared data


### 📁 Folder Structure After Setup
```
My Drive/
└── Datasets/
    └── Wearable-Biomechanics/
        ├── index/
        │   ├── entities_subjects.json
        │   └── entries_experiments.json
        ├── raw_data/
        │   └── experiments/
        └── related_files/
        │    └── experiments/
        └── processed_data/
            └── experiments/
```


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git
!pip install -q pygwalker

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local", # Use "local" since you are mounting Google Drive via Colab directly
    entity_id_format="sub_{num:04d}", # You can drop the `f""` string formatting here now
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [ ]:
# @title Storage Summary
store.storage_summary(by_folder=True, depth=2, by_file_type=True)


# **👤 People Management (Subject)**

In [ ]:
# @title 📝 Register New Subject { display-mode: "form" }
# @markdown Enter the details of the test subject to generate a unique ID.

name = "Dinithi" # @param {type:"string"}
age = 23 # @param {type:"number"}
weight_kg = 0 # @param {type:"number"}
height_m = 0 # @param {type:"number"}
achievement = "-"# @param {type:"string"}
best_weight = "-"# @param {type:"string"}
experience_years = 1 # @param {type:"integer"}
notes = "-" # @param {type:"string"}

# Logic to create entity
sub_id = store.create_entity(
    entity_type="subjects",
    name=name,
    age=age,
    weight=weight_kg,
    height=height_m,
    experience=experience_years,
    achievement=achievement,
    best_weight=best_weight,
    notes=notes
)

print(f"✅ Created Subject ID: {sub_id}")

In [ ]:
# @title 📋 Participant Registry { display-mode: "form" }
# @markdown View and filter the registered test subjects.

from google.colab import data_table

# Fetch subject data
df_subjects = store.list_entities("subjects")

if not df_subjects.empty:
    # 1. Clean up column names for display (e.g., replace underscores with spaces)
    df_subjects.columns = [c.replace('_', ' ').title() for c in df_subjects.columns]

    # 2. Display using Colab's Interactive Data Table
    print(f"Total Subjects Registered: {len(df_subjects)}")
    display(data_table.DataTable(
        df_subjects,
        include_index=False,
        num_rows_per_page=10,
        max_columns=20
    ))
else:
    print("⚠️ No participants found in the registry.")

# **📂 Experiment Upload & Data Exploration**

In [ ]:
# @title 🚀 Upload New Experiment { display-mode: "form" }
# @markdown Select a subject and upload their raw IMU data.

target_subject_id = "sub_0019" # @param {type:"string"}
activity_type = "Clean & Jerk" # @param {type:"string"}
weight_used = 30 # @param {type:"number"}
date = '2026-11-03' # @param {type:"string"}
time = '12:27' # @param {type:"string"}
timestamp = 1793689041 # @param {type:"number"}

notes = '3 reps, Complete ' # @param {type:"string"}

exp_id = store.create_entry_interactive(
    entry_type="experiments",
    entity_id=target_subject_id,
    date=date,
    time=time,
    description=activity_type,
    timestamp=timestamp,
    weight_kg=weight_used,
    notes=notes
)

# @markdown ---
# @markdown ### 📈 Auto-Visualization
# @markdown After upload, the IMU data will be plotted below for quality check.

if exp_id:
    raw_df = store.get_entry_raw_data("experiments", exp_id)

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Dynamically find Accelerometer columns (assuming AX, AY, AZ naming)
    accel_cols = [c for c in raw_df.columns if '_AX' in c or '_AY' in c or '_AZ' in c][:3]

    fig = go.Figure()
    for col in accel_cols:
        fig.add_trace(go.Scatter(y=raw_df[col], name=col, mode='lines'))

    fig.update_layout(
        title=f"Initial Data Check: {exp_id}",
        xaxis_title="Samples",
        yaxis_title="Acceleration ($m/s^2$)",
        template="plotly_dark",
        height=400
    )
    fig.show()

In [ ]:
# @markdown Browse all uploaded biomechanical data sessions.

import pygwalker as pyg
import ipywidgets as widgets
from IPython.display import display

df_experiments = store.list_entries("experiments",include_paths=True)

if not df_experiments.empty:
    df_experiments = df_experiments.sort_values(by="entry_id", ascending=False)
    df_experiments.columns = [c.replace('_', ' ').title() for c in df_experiments.columns]
    all_cols = list(df_experiments.columns)

    # 1. Create a checkbox for each column
    checkboxes = [
        widgets.Checkbox(
            value=True,
            description=col,
            indent=False,
            layout=widgets.Layout(width='auto', margin='2px 12px 2px 0')
        )
        for col in all_cols
    ]

    # 2. Quick-action buttons
    btn_all = widgets.Button(description="Select All", button_style="info", layout=widgets.Layout(width='100px'))
    btn_none = widgets.Button(description="Clear All", button_style="warning", layout=widgets.Layout(width='100px'))
    btn_apply = widgets.Button(description="Update Table", button_style="success", icon="refresh", layout=widgets.Layout(width='140px'))

    # 3. Arrange buttons and checkboxes into a clean grid
    button_bar = widgets.HBox([btn_all, btn_none, btn_apply], layout=widgets.Layout(margin='0 0 10px 0'))
    checkbox_grid = widgets.GridBox(
        checkboxes,
        layout=widgets.Layout(grid_template_columns="repeat(auto-fill, minmax(180px, 1fr))")
    )

    # Wrap inside a collapsible Accordion so it stays out of the way when not needed
    control_panel = widgets.Accordion(children=[widgets.VBox([button_bar, checkbox_grid])])
    control_panel.set_title(0, f"⚙️ Select Columns to Display ({len(all_cols)}/{len(all_cols)} selected)")
    control_panel.selected_index = None  # Starts collapsed to save screen space (set to 0 to start open)

    out = widgets.Output()

    # 4. Button logic
    def render_table(_=None):
        selected = [cb.description for cb in checkboxes if cb.value]
        control_panel.set_title(0, f"⚙️ Select Columns to Display ({len(selected)}/{len(all_cols)} selected)")
        with out:
            out.clear_output(wait=True)
            if selected:
                pyg.walk(
                    df_experiments[selected],
                    default_tab="data",
                    appearance="dark"
                )
            else:
                print("⚠️ Please select at least one column and click 'Update Table'.")

    def select_all(_):
        for cb in checkboxes:
            cb.value = True

    def select_none(_):
        for cb in checkboxes:
            cb.value = False

    btn_all.on_click(select_all)
    btn_none.on_click(select_none)
    btn_apply.on_click(render_table)

    # Display the UI and load the initial table
    display(control_panel, out)
    render_table()

else:
    print("No experiments recorded yet.")

In [73]:
store.list_entries("experiments",include_paths=True)

,entry_id,entity_id,timestamp,description,date,time,weight_kg,notes,raw_file_available,raw_file_path,...,rel_side_annotated_video_available,rel_side_annotated_video_path,rel_side_video_available,rel_side_video_path,rel_synced_video_available,rel_synced_video_path,video_sync_side_lag_frames,video_sync_front_delta_t_frames,video_sync_front_delta_t_imu_samples,video_sync_front_delta_t_sec
0,experiments_0001,sub_0001,1778229520,Clean & Jerk,2026-05-08,14:08,20,"5 reps, Sensor detached - left ankle last 3 reps",Available,raw_data/experiments/experiments_0001_sub_0001...,...,Available,related_files/experiments/experiments_0001/exp...,Available,related_files/experiments/experiments_0001/exp...,Available,related_files/experiments/experiments_0001/exp...,89.0,115.0,383.0,3.8333
1,experiments_0002,sub_0001,1778230197,Clean & Jerk,2026-05-08,14:20,20,"5 reps, Sensor detached - left ankle in rep 3",Available,raw_data/experiments/experiments_0002_sub_0001...,...,Available,related_files/experiments/experiments_0002/exp...,Available,related_files/experiments/experiments_0002/exp...,Available,related_files/experiments/experiments_0002/exp...,35.0,153.0,510.0,5.1000
2,experiments_0003,sub_0002,1778666861,Clean & Jerk,2026-05-13,15:37,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",Available,raw_data/experiments/experiments_0003_sub_0002...,...,Available,related_files/experiments/experiments_0003/exp...,Available,related_files/experiments/experiments_0003/exp...,Available,related_files/experiments/experiments_0003/exp...,76.0,-49.0,-163.0,-1.6333
3,experiments_0004,sub_0002,1778667119,Clean & Jerk,2026-05-13,15:42,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",Available,raw_data/experiments/experiments_0004_sub_0002...,...,Available,related_files/experiments/experiments_0004/exp...,Available,related_files/experiments/experiments_0004/exp...,Available,related_files/experiments/experiments_0004/exp...,32.0,-54.0,-180.0,-1.8000
4,experiments_0005,sub_0003,1778670186,Clean & Jerk,2026-05-13,16:33,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",Available,raw_data/experiments/experiments_0005_sub_0003...,...,Available,related_files/experiments/experiments_0005/exp...,Available,related_files/experiments/experiments_0005/exp...,Available,related_files/experiments/experiments_0005/exp...,-4.0,81.0,270.0,2.7000
5,experiments_0006,sub_0003,1778671087,Clean & Jerk,2026-05-13,16:48,20,"5 reps, Complete",Available,raw_data/experiments/experiments_0006_sub_0003...,...,Available,related_files/experiments/experiments_0006/exp...,Available,related_files/experiments/experiments_0006/exp...,Available,related_files/experiments/experiments_0006/exp...,37.0,-42.0,-140.0,-1.4000
6,experiments_0007,sub_0004,1781068783,Clean & Jerk,2026-06-10,10:49,20,"5 reps, Complete",Available,raw_data/experiments/experiments_0007_sub_0004...,...,Available,related_files/experiments/experiments_0007/exp...,Available,related_files/experiments/experiments_0007/exp...,Available,related_files/experiments/experiments_0007/exp...,-37.0,NaN,NaN,NaN
7,experiments_0008,sub_0004,1781069072,Clean & Jerk,2026-06-10,10:54,20,"5 reps, Complete",Available,raw_data/experiments/experiments_0008_sub_0004...,...,Available,related_files/experiments/experiments_0008/exp...,Available,related_files/experiments/experiments_0008/exp...,Available,related_files/experiments/experiments_0008/exp...,15.0,-66.0,-220.0,-2.2000
8,experiments_0009,sub_0004,1781069235,Clean & Jerk,2026-06-10,10:57,20,"5 reps, Complete",Available,raw_data/experiments/experiments_0009_sub_0004...,...,Available,related_files/experiments/experiments_0009/exp...,Available,related_files/experiments/experiments_0009/exp...,Available,related_files/experiments/experiments_0009/exp...,-15.0,-97.0,-323.0,-3.2333
9,experiments_0010,sub_0004,1781069666,Clean & Jerk,2026-06-10,11:04,20,"5 reps, Complete",Available,raw_data/experiments/experiments_0010_sub_0004...,...,Available,related_files/experiments/experiments_0010/exp...,A

In [ ]:
# @title 📎 Attachment Manager { display-mode: "form" }
# @markdown Upload related files (videos, high-speed clips, or archives) to an experiment.

experiment_to_attach = "experiments_0060" # @param {type:"string"}
file_label = "side_video" # @param ["front_video", "side_video", "raw_zip", "other"]

rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=experiment_to_attach,
    label=file_label
)

# **📊  Advanced IMU Analysis Plot**

In [ ]:
# @title
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.io as pio
pio.renderers.default = "colab"

def get_columns(df, devices=None, imus=None, axes=None):
    """
    Extracts sensor columns matching devices, imus, and axes.
    If devices, imus, or axes are None, it matches all available.
    """
    if devices is None:
        devices = [r"\d+"]
    elif not isinstance(devices, list):
        devices = [devices]

    if imus is None:
        imus = [r"\d+"]
    elif not isinstance(imus, list):
        imus = [imus]

    if axes is None:
        axes = [r"[A-Z0-9]+"]
    elif not isinstance(axes, list):
        axes = [axes]

    valid_columns = []
    for dev in devices:
        for imu in imus:
            for axis in axes:
                pattern = rf"^.*?(?<!\d){dev}_IMU{imu}_{axis}$"
                for col in df.columns:
                    if re.match(pattern, col) and col not in valid_columns:
                        valid_columns.append(col)
    return valid_columns


# =====================================================================
# 1. MACRO VERIFICATION: Health Check & Anomaly Summary
# =====================================================================
def get_sensor_health_summary(df):
    """
    Scans every IMU channel to detect:
    - Dead / Flatlined sensors (std == 0)
    - Missing values (NaN count)
    - Extreme clipping / out-of-range sensor readings
    """
    pattern = re.compile(r"^.*?(?<!\d)(\d+)_IMU(\d+)_(AX|AY|AZ|GX|GY|GZ)$")
    records = []

    for col in df.columns:
        m = pattern.match(col)
        if not m:
            continue
        dev, imu, axis = int(m.group(1)), int(m.group(2)), m.group(3)
        series = df[col]

        s_min, s_max, s_std = series.min(), series.max(), series.std()
        nan_count = series.isna().sum()
        is_flat = bool(s_std == 0 or series.nunique() <= 1)

        # Standard sanity bounds: Accel |val| > 80 m/s^2 (~8g), Gyro |val| > 2000 deg/s
        is_acc = axis.startswith("A")
        clipped = bool((s_min < -80 or s_max > 80) if is_acc else (s_min < -2000 or s_max > 2000))

        if is_flat:
            status = "FLATLINE"
        elif nan_count > 0:
            status = "NANS"
        elif clipped:
            status = "CLIPPED"
        else:
            status = "OK"

        records.append({
            "Device": f"Dev {dev}",
            "Channel": f"IMU{imu}_{axis}",
            "Status": status,
            "Std": s_std,
            "Min": s_min,
            "Max": s_max,
            "Column": col
        })

    return pd.DataFrame(records)


def plot_health_matrix(df, title="Sensor Channel Quality Matrix"):
    """
    Visualizes all 150+ channels on a single status matrix.
    Instantly highlights dead (0.0) sensors, clipping, or missing data.
    """
    health_df = get_sensor_health_summary(df)
    if health_df.empty:
        print("No matching IMU channels detected.")
        return

    # Map status to categorical score for color mapping
    status_map = {"OK": 0, "CLIPPED": 1, "NANS": 2, "FLATLINE": 3}
    health_df["Score"] = health_df["Status"].map(status_map)

    pivot = health_df.pivot(index="Device", columns="Channel", values="Score")

    # Sort devices naturally (Dev 1, Dev 2, ..., Dev 10)
    pivot = pivot.reindex(sorted(pivot.index, key=lambda x: int(x.split()[1])))

    plt.figure(figsize=(18, 5))
    cmap = sns.color_palette(["#2ecc71", "#f39c12", "#e67e22", "#e74c3c"])
    ax = sns.heatmap(pivot, cmap=cmap, cbar=False, linewidths=0.5, linecolor="#eeeeee")

    plt.title(f"{title} (Green = OK, Orange = Clipped, Red = Flatlined/Dead)", fontsize=13)
    plt.xlabel("IMU Axis Channels")
    plt.ylabel("Devices")
    plt.xticks(rotation=90, fontsize=8)
    plt.tight_layout()
    plt.show()


# =====================================================================
# 2. DEEP-DIVE VERIFICATION: Interactive Plotly Dashboard
# =====================================================================
def plot_imu_verification_dashboard(df, time_col="Timestamp"):
    """
    Builds an interactive Plotly dashboard featuring:
    - Dropdown switcher for Device 1 to 10
    - Synchronized time series subplots for Accelerometer (m/s^2) and Gyroscope (deg/s)
    - Side-by-side Violin/Box distribution plots to detect outliers and drift
    """
    # Detect all present devices
    pattern = re.compile(r"^.*?(?<!\d)(\d+)_IMU(\d+)_(AX|AY|AZ|GX|GY|GZ)$")
    devices = sorted(list({int(pattern.match(c).group(1)) for c in df.columns if pattern.match(c)}))

    if not devices:
        raise ValueError("No matching device columns found in dataframe.")

    # 4 Subplots: Accel Time-Series, Accel Violin, Gyro Time-Series, Gyro Violin
    fig = make_subplots(
        rows=2, cols=2,
        column_widths=[0.72, 0.28],
        shared_xaxes=False,
        vertical_spacing=0.1,
        horizontal_spacing=0.06,
        subplot_titles=(
            "Accelerometer Signals (AX, AY, AZ)", "Accelerometer Distribution",
            "Gyroscope Signals (GX, GY, GZ)", "Gyroscope Distribution"
        )
    )

    t = df[time_col] if time_col in df.columns else np.arange(len(df))

    # Pre-generate traces for all devices and toggle visibility with dropdown
    device_trace_counts = []

    for dev in devices:
        count = 0
        acc_cols = get_columns(df, devices=dev, axes=["AX", "AY", "AZ"])
        gyro_cols = get_columns(df, devices=dev, axes=["GX", "GY", "GZ"])

        # Top-Left: Accel Time-Series
        for col in acc_cols:
            fig.add_trace(
                go.Scatter(x=t, y=df[col], name=col.split(f"{dev}_")[-1], mode="lines",
                           line=dict(width=1.2), legendgroup=f"dev_{dev}", visible=(dev == devices[0])),
                row=1, col=1
            )
            count += 1

        # Top-Right: Accel Violin / Distribution
        for col in acc_cols:
            fig.add_trace(
                go.Violin(y=df[col], name=col.split(f"{dev}_")[-1], box_visible=True,
                          meanline_visible=True, legendgroup=f"dev_{dev}", showlegend=False,
                          visible=(dev == devices[0])),
                row=1, col=2
            )
            count += 1

        # Bottom-Left: Gyro Time-Series
        for col in gyro_cols:
            fig.add_trace(
                go.Scatter(x=t, y=df[col], name=col.split(f"{dev}_")[-1], mode="lines",
                           line=dict(width=1.2), legendgroup=f"dev_{dev}", visible=(dev == devices[0])),
                row=2, col=1
            )
            count += 1

        # Bottom-Right: Gyro Violin / Distribution
        for col in gyro_cols:
            fig.add_trace(
                go.Violin(y=df[col], name=col.split(f"{dev}_")[-1], box_visible=True,
                          meanline_visible=True, legendgroup=f"dev_{dev}", showlegend=False,
                          visible=(dev == devices[0])),
                row=2, col=2
            )
            count += 1

        device_trace_counts.append(count)

    # Build Dropdown Menu
    buttons = []
    cum_traces = [0] + list(np.cumsum(device_trace_counts))

    for i, dev in enumerate(devices):
        visibility = [False] * cum_traces[-1]
        start_idx = cum_traces[i]
        end_idx = cum_traces[i + 1]
        for idx in range(start_idx, end_idx):
            visibility[idx] = True

        buttons.append(dict(
            label=f"Device {dev}",
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"IMU Verification Dashboard — Device {dev}"}
            ]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.0, y=1.15,
            xanchor="left", yanchor="top",
            direction="down",
            showactive=True
        )],
        title=dict(text=f"IMU Verification Dashboard — Device {devices[0]}", x=0.5, y=0.98),
        height=850,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(orientation="h", y=-0.12, x=0.0)
    )

    fig.update_yaxes(title_text="Accel (m/s²)", row=1, col=1)
    fig.update_yaxes(title_text="Value Range", row=1, col=2)
    fig.update_yaxes(title_text="Gyro (°/s)", row=2, col=1)
    fig.update_yaxes(title_text="Value Range", row=2, col=2)
    fig.update_xaxes(title_text="Timestamp / Sample", row=2, col=1)

    return fig

In [ ]:
# @title 🔍 Detailed Signal Analysis { display-mode: "form" }
# @markdown Compare different IMU nodes on the body.

selected_exp = "experiments_0055" # @param {type:"string"}
df_analysis = store.get_entry_raw_data("experiments", selected_exp)

plot_health_matrix(df_analysis, title="Trial 0001 Sensor Health Matrix")

fig = plot_imu_verification_dashboard(df_analysis, time_col="Timestamp")
fig.show()